# Anchored Agent — Calm 2025 vs. Shock 2026

`anchor_2026_eval_scorecard.ipynb` measured the anchored agent on one window: the
shock-heavy 2026 eval, 18 origins. Everything concluded there — that the anchored
agent beats the bare anchor, that `w_loc` wants to be larger, that the bounded
schema suppresses downward views — rested on that single regime.

This notebook adds the calm 2025 backtest (51 origins) and asks what survives.

**The short answer: almost nothing, and the one thing that does is not a property
of the agent.**

1. The anchored agent's effect on CRPS **flips sign** between the two windows, and
   is statistically significant *in both directions*.
2. `w_loc`'s optimum sits at **opposite ends of the grid** — 0.0 on 2025, 1.0 on 2026.
3. The width channel's contribution is **exactly reproduced by a constant**, in
   both regimes, in every arm.
4. Decomposing the news baseline's advantage on 2025: its edge is its **narrower
   prediction intervals**, not its news reading. The news reasoning itself
   *significantly worsens* the score on the weaker model.

Full write-up: `planning-docs/anchored-agent-cross-regime-findings.md`.
All analysis replays stored predictions — **no LLM calls in this notebook.**

In [1]:
import warnings

import numpy as np
import pandas as pd

from energy_oil_forecasting.anchor_signal_analysis import (
    ARMS,
    DEFAULT_GRID,
    MODELS,
    bootstrap_ci,
    channel_ablation,
    channel_control,
    contribution_ci,
    decompose_baseline,
    directional_edge,
    gate_table,
    load_arm,
    naive_signal,
    score,
    sign_only_control,
    trailing_vol,
    weight_sweep,
)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)

WINDOWS = {"2025 calm": "energy_oil_backtest", "2026 shock": "energy_oil_eval"}

# Fetched once and threaded through every call -- each load otherwise re-reads the
# whole close series.
from energy_oil_forecasting.anchor_signal_analysis import _actuals  # noqa: E402

ACTUALS = _actuals()
print(f"{len(ACTUALS)} realised closes loaded")

5680 realised closes loaded


## 1. What actually ran

**Updated 2026-08-09**: the anchor-gap fix landed (`AnchorSource.available_horizons()`
+ changes in `AnchoredWtiPromptBuilder`/`_reconstruct_predictions` — see
`planning-docs/news-cache-rebuild-interview-notes.md`). It used to be true that *"the
anchored predictor needs an anchor for every horizon before it can even build its
prompt, so a single missing horizon costs it the whole origin"* — that is no longer
the case. A missing horizon now only drops that horizon; the origin's other horizons
still resolve. The cell counts below reflect the fixed behavior (anchored arms lose
only the individual missing-horizon cells, not whole origins) — before the fix,
2026's `original`/`symloc` arms resolved 42 cells (14/18 origins), not 50.

Those missing horizons are still **market holidays** — `2026-02-16` is Presidents'
Day, `2026-05-25` is Memorial Day. Horizons step by pandas business days, which do
not know about them, so `origin + 5B` and `origin + 10B` land on a closed market. The
anchor table genuinely cannot hold those entries: there is no close to score
against. That root cause is unchanged; what changed is only how much of an origin
a missing horizon takes down with it.</cell>


In [2]:
rows = []
for wlabel, spec in WINDOWS.items():
    for arm in ARMS:
        for model in MODELS:
            d = load_arm(spec, arm, model, ACTUALS)
            rows.append({
                "window": wlabel, "arm": arm, "model": model,
                "cells": 0 if d is None else len(d),
                "origins": 0 if d is None else d.index.get_level_values(0).nunique(),
            })
grid = pd.DataFrame(rows).pivot_table(
    index=["window", "arm"], columns="model", values=["cells", "origins"], fill_value=0)
grid

cells           origins        
model                3.5-flash preview 3.5-flash preview
window     arm                                          
2025 calm  free-form     145.0   145.0      51.0    51.0
           original      145.0   145.0      51.0    51.0
           symloc        145.0   145.0      51.0    51.0
           twosided      145.0   142.0      51.0    50.0
2026 shock free-form      50.0    50.0      18.0    18.0
           original       50.0    50.0      18.0    18.0
           symloc         50.0    50.0      18.0    18.0
           twosided       50.0    50.0      18.0    18.0

## 2. THE HEADLINE — where does the news baseline's advantage come from?

On 2025 the news baseline beats bare AutoARIMA. That much is real. The question is
which *part* of the agent earns it.

Four forecasts, each adding exactly one ingredient, scored on the same cells:

| | centre | interval width | adds |
|---|---|---|---|
| **A** | AutoARIMA's point | AutoARIMA's | nothing — no agent at all |
| **B** | the origin day's close | AutoARIMA's | the price level the news states |
| **C** | the origin day's close | **the agent's** | the agent's uncertainty sizing |
| **D** | **the agent's** point | **the agent's** | its actual news judgement |

`D − C` is the question the whole project rests on: **what does the news reasoning
add beyond restating a price the briefing already gave away?**

Why B exists at all: the price *data* stops at the previous close — correctly, since
standing at the origin you do not know that day's close, and the anchor is fit on
exactly that. But the news briefing has cutoff `as_of` and routinely states the
current level. That is a legitimate edge, not lookahead, but it is not *reasoning*,
so it has to be separated out before crediting the agent with anything.

In [3]:
out = []
for wlabel, spec in WINDOWS.items():
    for model in MODELS:
        d = decompose_baseline(spec, model, ACTUALS)
        if d.empty:
            continue
        row = {"window": wlabel, "model": model, "origins": d["as_of"].nunique(), "n": len(d)}
        row |= {k: round(d[k].mean(), 3) for k in "ABCD"}
        for label, a, b in [("① price level", "B", "A"),
                            ("② interval width", "C", "B"),
                            ("③ news reasoning", "D", "C")]:
            mu, lo, hi = contribution_ci(d, a, b)
            row[label] = f"{mu:+.3f} [{lo:+.3f}, {hi:+.3f}]" + ("  *" if not lo <= 0 <= hi else "")
        out.append(row)
pd.DataFrame(out).set_index(["window", "model"])

origins    n      A      B      C      D            ① price level            ② interval width  \
window     model                                                                                                      
2025 calm  preview         51  145  2.285  2.185  1.949  2.203  -0.100 [-0.293, +0.066]  -0.236 [-0.398, -0.047]  *   
           3.5-flash       51  145  2.285  2.185  1.886  2.002  -0.100 [-0.293, +0.066]  -0.299 [-0.397, -0.192]  *   
2026 shock preview         18   50  9.089  9.137  9.536  9.555  +0.049 [-1.183, +1.293]     +0.399 [-0.015, +0.840]   
           3.5-flash       18   50  9.089  9.137  8.973  9.159  +0.049 [-1.183, +1.293]     -0.164 [-0.841, +0.441]   

                             ③ news reasoning  
window     model                               
2025 calm  preview    +0.254 [-0.014, +0.515]  
           3.5-flash  +0.116 [-0.116, +0.351]  
2026 shock preview    +0.019 [-1.177, +1.195]  
           3.5-flash  +0.186 [-1.162, +1.688]

**Reading the 2025 rows** (51 origins — the only window with enough power to say
anything; 2026's intervals here are ~±1.2 wide and answer nothing):

- **② is the only significant positive contribution.** The agent uses *narrower
  prediction intervals* than AutoARIMA, and in a calm year narrow intervals win.
- **③ is no longer significant on `preview`, after the 2026-08-09 news-cache
  rebuild.** Before the rebuild this read "+0.308, significantly harmful." The
  rebuilt cache (aligned cutoff, no more raw-deliberation leaks) now gives
  **+0.254 [-0.014, +0.515]** — smaller, and the CI now straddles zero. Still
  positive-but-not-significant on `3.5-flash` (was +0.168, now +0.116
  [-0.116, +0.351]). Read this as "no established harm from news reasoning on
  2025," not as "confirmed harmless" — the point estimate is still positive on
  both models, it just no longer clears the bar this notebook uses elsewhere.
- **① is not significant.** Knowing the level the news states is worth about −0.10
  as a point estimate, but the interval straddles zero.

Concretely: take today's price plus the agent's interval width, **throw its point
forecast away entirely**, and you score **1.904**. Add the point forecast back and
you get **2.212**.

And ② is itself a *regime bet*, not skill — on 2026 the same contribution is +0.230
(preview, hurts) and −0.187 (3.5-flash, helps). AutoARIMA's intervals are too wide
when calm and too narrow in shocks; the agent happens to sit narrow. A
volatility-scaled interval would achieve this with a principle instead of a
coincidence.</cell>


## 3. The sign flips between regimes — significantly, in both directions

`anchored agent (w_loc=0.2, w_width=0.5) − anchor alone`. Negative means the agent
helps.

In [4]:
rows = []
for wlabel, spec in WINDOWS.items():
    for model in MODELS:
        d = load_arm(spec, "original", model, ACTUALS)
        if d is None:
            continue
        mu, lo, hi = bootstrap_ci(d, (0.2, 0.5), (0.0, 0.0))
        rows.append({
            "window": wlabel, "model": model, "n": len(d),
            "anchored − anchor": round(mu, 3),
            "95% CI": f"[{lo:+.3f}, {hi:+.3f}]",
            "verdict": ("agent WORSE *" if lo > 0 else "agent better *" if hi < 0 else "n.s."),
        })
pd.DataFrame(rows).set_index(["window", "model"])

n  anchored − anchor            95% CI         verdict
window     model                                                              
2025 calm  preview    145              0.129  [+0.094, +0.168]   agent WORSE *
           3.5-flash  145              0.059  [+0.026, +0.091]   agent WORSE *
2026 shock preview     50             -0.181  [-0.325, -0.040]  agent better *
           3.5-flash   50             -0.114  [-0.269, +0.030]            n.s.

The agent helps in the shock window and **hurts in the calm one** — and 2025 has
three times the data. On weight of evidence the anchored agent is not helping.

### 3.1 `w_loc` wants opposite extremes

If a single weight were right, both windows would point the same way.

In [5]:
sweep = {}
for wlabel, spec in WINDOWS.items():
    for model in MODELS:
        d = load_arm(spec, "original", model, ACTUALS)
        if d is not None:
            sweep[(wlabel, model)] = weight_sweep(d)
sw = pd.DataFrame(sweep).T.round(3)
sw["argmin"] = [DEFAULT_GRID[int(np.argmin(r))] for r in sw.values]
sw

w_loc                   0.0    0.1    0.2    0.3    0.5    0.7    1.0  argmin
2025 calm  preview    2.388  2.397  2.414  2.437  2.509  2.611  2.815     0.0
           3.5-flash  2.342  2.341  2.344  2.350  2.372  2.407  2.485     0.1
2026 shock preview    8.886  8.894  8.907  8.923  8.968  9.030  9.153     0.0
           3.5-flash  8.932  8.951  8.974  9.000  9.058  9.133  9.273     0.0

**Opposite ends of the grid.** Fitting `w_loc` on one window fits that window's
direction, not the agent's reliability.

Why 2026 pushes it to 1.0: the truth sat above the anchor 60% of the time and the
agent's signal was mostly positive, so raising `w_loc` amplifies a bullish tilt in
a rising market. A *constant* positive signal at `w_loc=1.0` improves 2026's CRPS
by ~0.65 all by itself.

**A monotone sweep running to the edge of the grid is a warning sign, not a
recommendation.**

## 4. The width channel is a constant — the one result that replicates

Two tests, in order. First the usual one: is the channel significant?

In [6]:
rows = []
for wlabel, spec in WINDOWS.items():
    for model in MODELS:
        d = load_arm(spec, "original", model, ACTUALS)
        if d is None:
            continue
        mu, lo, hi = bootstrap_ci(d, (0.2, 0.5), (0.2, 0.0))   # full vs location-only
        rows.append({"window": wlabel, "model": model,
                     "width channel": round(mu, 3), "95% CI": f"[{lo:+.3f}, {hi:+.3f}]",
                     "significant": "yes" if not lo <= 0 <= hi else "no"})
pd.DataFrame(rows).set_index(["window", "model"])

width channel            95% CI significant
window     model                                                 
2025 calm  preview            0.102  [+0.081, +0.124]         yes
           3.5-flash          0.059  [+0.046, +0.072]         yes
2026 shock preview           -0.200  [-0.320, -0.094]         yes
           3.5-flash         -0.150  [-0.225, -0.081]         yes

By the usual standard the agent's width judgement is doing real work.

Now the control: replace every cell's `signal_width` with **the mean of that arm's
own `signal_width`** — deleting all per-cell judgement, keeping only the average
widening. If the agent's judgement matters, this should score worse.

In [7]:
rows = []
for wlabel, spec in WINDOWS.items():
    for model in MODELS:
        d = load_arm(spec, "original", model, ACTUALS)
        if d is None:
            continue
        c = channel_control(d, "width")
        rows.append({"window": wlabel, "model": model,
                     "real": round(c["real"], 3), "constant": round(c["constant"], 3),
                     "shuffled": round(c["shuffled"], 3),
                     "real − constant": round(c["real_minus_constant"], 3)})
pd.DataFrame(rows).set_index(["window", "model"])

real  constant  shuffled  real − constant
window     model                                                
2025 calm  preview    2.414     2.398     2.399            0.016
           3.5-flash  2.344     2.340     2.340            0.004
2026 shock preview    8.907     8.883     8.883            0.024
           3.5-flash  8.974     8.950     8.957            0.024

**Identical — in both regimes, in every arm.** The significant channel is a constant
recalibration of AutoARIMA's intervals. The agent's per-cell width judgement
contributes nothing.

This is the only finding here that replicates cleanly across both regimes, and it
is not a finding about agent capability. Actionable version: **AutoARIMA's
prediction intervals are miscalibrated — widen them by a fixed factor, or better one
that scales with recent volatility. No LLM required.**

### 4.1 The location channel's magnitude adds nothing either

Same idea, applied to `signal_loc`: keep only its **sign** and rescale `w_loc` by
`mean|signal_loc|` so the average dollar move is unchanged. Without that rescaling
the sign-only version would move several times further and the test would measure
size rather than information.

In [8]:
rows = []
for wlabel, spec in WINDOWS.items():
    for arm in ("original", "symloc", "twosided"):
        for model in MODELS:
            d = load_arm(spec, arm, model, ACTUALS)
            if d is None:
                continue
            c = sign_only_control(d)
            rows.append({"window": wlabel, "arm": arm, "model": model,
                         "real": round(c["real"], 3), "sign only": round(c["sign_only"], 3),
                         "cost of dropping magnitude": round(c["sign_minus_real"], 3)})
pd.DataFrame(rows).set_index(["window", "arm", "model"])

real  sign only  cost of dropping magnitude
window     arm      model                                                  
2025 calm  original preview    2.414      2.405                      -0.009
                    3.5-flash  2.344      2.341                      -0.003
           symloc   preview    2.434      2.426                      -0.008
                    3.5-flash  2.335      2.334                      -0.001
           twosided preview    2.397      2.397                       0.000
                    3.5-flash  2.350      2.346                      -0.004
2026 shock original preview    8.907      8.870                      -0.037
                    3.5-flash  8.974      8.934                      -0.041
           symloc   preview    8.879      8.820                      -0.059
                    3.5-flash  9.005      8.960                      -0.045
           twosided preview    8.888      8.810                      -0.078
                    3.5-flash  8.873      8.891                       0.018

Discarding the agent's magnitude costs essentially nothing. That licenses a much
simpler contract — the agent emits ±1 and the harness owns the magnitude — which
also removes its ability to hedge at +0.1, something the anchored arms did
constantly.

It does **not** make the regime problem go away; it concentrates it, since `w_loc`
would then carry the entire magnitude.

## 5. Volatility does not explain the advantage

The natural rescue for section 3: perhaps the agent is genuinely better when
volatility is high, so `w_loc` should scale with it. Pooling both windows, this
looks strongly true.

In [9]:
frames = []
for wlabel, spec in WINDOWS.items():
    for model in MODELS:
        d = load_arm(spec, "original", model, ACTUALS)
        if d is None:
            continue
        r = d.reset_index()
        r["advantage"] = score(d, 0.0, 0.0) - score(d, 0.2, 0.5)   # >0 = agent helped
        r["loc_only_advantage"] = score(d, 0.0, 0.5) - score(d, 0.2, 0.5)
        r["vol"] = [trailing_vol(o, ACTUALS) for o in r["as_of"]]
        r["window"] = wlabel
        frames.append(r)
vol = pd.concat(frames)
# thresholds are the ones the adaptive agent's SKILL.md actually reasons with
vol["bucket"] = pd.cut(vol["vol"], [0, 20, 35, 55, 999],
                       labels=["low <20", "normal 20-35", "elevated 35-55", "extreme >55"])


def bucket_table(df):
    return df.groupby("bucket", observed=True).agg(
        n=("advantage", "size"), origins=("as_of", "nunique"),
        mean_vol=("vol", "mean"), agent_advantage=("advantage", "mean"),
        loc_only=("loc_only_advantage", "mean")).round(3)


print("POOLED across both windows:")
bucket_table(vol)

POOLED across both windows:


,n,origins,mean_vol,agent_advantage,loc_only
bucket,,,,,
low <20,10,2,19.058,-0.001,0.091
normal 20-35,216,38,25.438,-0.096,-0.021
elevated 35-55,100,18,44.796,0.057,0.020
extreme >55,64,11,86.250,0.039,-0.089


A clean monotone relationship — advantage rises with volatility.

**Now the control.** If volatility is really the driver, the pattern has to hold
*within a single window*, where "volatile" is not a proxy for "is 2026".

In [10]:
print("2025 ONLY — same buckets:")
bucket_table(vol[vol.window == "2025 calm"])

2025 ONLY — same buckets:


,n,origins,mean_vol,agent_advantage,loc_only
bucket,,,,,
low <20,10,2,19.058,-0.001,0.091
normal 20-35,216,38,25.438,-0.096,-0.021
elevated 35-55,64,11,46.546,-0.103,-0.008


The same `elevated 35-55` bucket is **positive when pooled and negative within 2025
alone**. The pooled effect is the *window* wearing volatility as a disguise.

**Learning `w_loc` from volatility would learn "is this 2026".** In a high-volatility
window that *fell*, such a rule would be actively dangerous — and we have no such
window to check against, which is precisely the gap.

One real signal does survive: in the `extreme` bucket almost the entire advantage is
the width channel, not location. That is consistent with section 4 — widen intervals
when volatility is high — and again needs no LLM.

**Caveat**: 2025 has no `extreme` bucket at all, so that row cannot be
within-window checked. The honest claim is *"within the range where a control is
possible, volatility does not explain it"*.

## 6. Label-free gate — and why the rectification story did not survive

The gate compares each anchored arm's signal distribution to the **same model's own
unanchored view** on the same cells. It never reads a realised price, so iterating
prompts against it cannot overfit the evaluation window.

`sd_ratio` targets 1.0 (the anchored format should not compress the model's range)
and `mean_gap` targets 0 (it should not manufacture a lean).

In [11]:
for wlabel, spec in WINDOWS.items():
    for model in MODELS:
        g = gate_table(spec, model, ACTUALS)
        if g.empty:
            continue
        print(f"\n=== {wlabel} / {model} ===")
        print(g.round(3).to_string())


=== 2025 calm / preview ===
             n   mean     sd  pct_negative  sd_ratio  mean_gap
arm                                                           
free-form  142 -0.050  0.101        66.901       NaN       NaN
original   142 -0.158  0.210        69.014     2.085    -0.107
symloc     142 -0.210  0.219        77.465     2.169    -0.160
twosided   142 -0.248  0.214        83.803     2.121    -0.198



=== 2025 calm / 3.5-flash ===
             n   mean     sd  pct_negative  sd_ratio  mean_gap
arm                                                           
free-form  145 -0.049  0.088        75.862       NaN       NaN
original   145 -0.098  0.140        73.103     1.595    -0.049
symloc     145 -0.143  0.172        82.069     1.963    -0.094
twosided   145 -0.142  0.141        84.828     1.608    -0.093



=== 2026 shock / preview ===
            n   mean     sd  pct_negative  sd_ratio  mean_gap
arm                                                          
free-form  50 -0.039  0.338          50.0       NaN       NaN
original   50  0.080  0.278          26.0     0.822     0.119
symloc     50  0.025  0.317          34.0     0.937     0.063
twosided   50  0.024  0.317          46.0     0.938     0.063



=== 2026 shock / 3.5-flash ===
            n   mean     sd  pct_negative  sd_ratio  mean_gap
arm                                                          
free-form  50 -0.041  0.250          56.0       NaN       NaN
original   50  0.014  0.246          40.0     0.986     0.055
symloc     50 -0.053  0.338          56.0     1.352    -0.012
twosided   50 -0.091  0.267          70.0     1.070    -0.051


**This is what retracted the previous session's headline.** On 2026 the anchored arm
was 2% negative where the baseline was 33%, with the spread crushed 2.5x — which
looked like the bounded schema rectifying away downward views.

On 2025 the anchored arm is **64–81% negative with more spread than the baseline**
(`sd_ratio` 1.86 for preview). Rectification is a property of the **2026 window**,
not of the schema.

### 6.1 And the "unexplained +0.1 lean" is mostly a one-day data lag

A zero-judgement agent that merely restates the origin day's close — the level the
news states and the daily-close data does not yet contain — already implies a
non-zero signal. Comparing each arm's mean against that separates "leans up because
of the news" from "the anchor is one day stale and the window rose".

In [12]:
rows = []
for wlabel, spec in WINDOWS.items():
    for arm in ARMS:
        for model in MODELS:
            d = load_arm(spec, arm, model, ACTUALS)
            if d is None:
                continue
            sig = d["signal_loc"].clip(-1, 1) if arm == "free-form" else d["signal_loc"]
            nv = naive_signal(d)
            rows.append({"window": wlabel, "arm": arm, "model": model,
                         "signal mean": round(sig.mean(), 3),
                         "zero-judgement baseline": round(float(nv.mean()), 3),
                         "excess": round(sig.mean() - float(nv.mean()), 3)})
pd.DataFrame(rows).set_index(["window", "arm", "model"])

signal mean  zero-judgement baseline  excess
window     arm       model                                                  
2025 calm  free-form preview         -0.052                   -0.003  -0.048
                     3.5-flash       -0.049                   -0.003  -0.046
           original  preview         -0.161                   -0.003  -0.157
                     3.5-flash       -0.098                   -0.003  -0.095
           symloc    preview         -0.212                   -0.003  -0.208
                     3.5-flash       -0.143                   -0.003  -0.140
           twosided  preview         -0.248                   -0.004  -0.243
                     3.5-flash       -0.142                   -0.003  -0.138
2026 shock free-form preview         -0.039                    0.119  -0.158
                     3.5-flash       -0.041                    0.119  -0.160
           original  preview          0.080                    0.119  -0.039
                     3.5-flash        0.014                    0.119  -0.105
           symloc    preview          0.025                    0.119  -0.095
                     3.5-flash       -0.053                    0.119  -0.172
           twosided  preview          0.024                    0.119  -0.095
                     3.5-flash       -0.091                    0.119  -0.211

For `3.5-flash / original` on 2026 the excess over a zero-judgement baseline is
**+0.006** — the entire "manufactured lean" is the lag. And the lean tracks the
window's drift across regimes (2025's baseline is ≈0), which is what the lag story
predicts and a psychological-anchoring story does not.

**Consequence**: if the anchor is structurally one day stale, the correct signal
centre is *not* zero. The `twosided` prompt pushed `3.5-flash` to ≈0 when a
zero-judgement baseline sits at +0.127 — it may have suppressed real information,
which would explain why every gate metric improved while CRPS did not move.

*Partial, not total*: the cell-level correlation between arms' signals and the naive
signal is ≈0, so the lag explains the **level**, not the **variation**. The spread
findings above are unaffected.

## 7. Directional skill, scored against the right null

`hit rate − null`, where the null is *"guess randomly with the same up/down
frequency this arm used"*. Not 0.5: an arm that says "up" 97% of the time in a
window that rose 60% of the time scores 0.60 while knowing nothing.

`origin_close` asks *"will the price rise from here?"* — the question a news agent
that never sees the anchor was actually asked. `anchor_pt` asks *"will it land above
the ARIMA forecast?"*, which is what `signal_loc` encodes. Scoring an unanchored arm
against `anchor_pt` measures a question it was never posed.

In [13]:
rows = []
for wlabel, spec in WINDOWS.items():
    for arm in ARMS:
        for model in MODELS:
            d = load_arm(spec, arm, model, ACTUALS)
            if d is None:
                continue
            for ref in ("origin_close", "anchor_pt"):
                e = directional_edge(d, reference=ref)
                rows.append({"window": wlabel, "arm": arm, "model": model,
                             "reference": ref, "n": int(e["n"]),
                             "says up": round(e["says_up"], 2),
                             "hit": round(e["hit_rate"], 2),
                             "null": round(e["null"], 2),
                             "edge": round(e["edge"], 3)})
pd.DataFrame(rows).set_index(["window", "arm", "model", "reference"]).unstack()["edge"]

reference                       anchor_pt  origin_close
window     arm       model                             
2025 calm  free-form 3.5-flash     -0.010        -0.045
                     preview       -0.076        -0.087
           original  3.5-flash     -0.041        -0.074
                     preview       -0.073        -0.063
           symloc    3.5-flash     -0.044        -0.059
                     preview       -0.055        -0.067
           twosided  3.5-flash     -0.030        -0.055
                     preview       -0.089        -0.082
2026 shock free-form 3.5-flash     -0.030         0.011
                     preview        0.020         0.128
           original  3.5-flash     -0.014         0.101
                     preview       -0.017         0.076
           symloc    3.5-flash     -0.030         0.058
                     preview        0.023         0.116
           twosided  3.5-flash      0.012         0.104
                     preview        0.014         0.079

No edge is established anywhere, and the larger window leans negative.

**Two things this does not prove.** It does not show the model cannot read news — it
shows news reading does not produce a directional edge *on WTI at 5–21 business
days*. And **market efficiency predicts exactly this**: WTI futures are highly
liquid and the briefing is public information already in the price, so a perfect
news reader should also show zero edge. That makes directional alpha the wrong
success criterion, since no method would clear it.

**Power**: confidence intervals here are ±0.10–0.19 wide, so an edge of ~+0.05 would
be invisible. With stride 5 and horizons out to 21, each origin's window overlaps
its four neighbours, so effective independence is well below the nominal origin
count. Correct reading: *a large edge is ruled out; a small one is not testable
here.*

## 8. What survives both regimes

**Established.**

- AutoARIMA's prediction intervals are miscalibrated, and a **constant** widening
  captures the entire gain the agent's width channel appears to deliver.
- The anchored agent's effect on CRPS **flips sign between regimes**, significantly
  in both directions.
- `w_loc`'s optimum sits at **opposite ends of the grid** in the two windows.
- On 2025 the news baseline's only significant positive contribution is its
  **narrower intervals**; its news reasoning's harm on preview is no longer
  significant after the 2026-08-09 news-cache rebuild (+0.308 → +0.254, CI now
  includes 0 — see §2's updated reading).
- Discarding the agent's location magnitude has **no measurable cost**.

**Not established.**

- Any small directional edge — ruled out above ~0.10, not testable below.
- Whether the `extreme` volatility bucket's advantage is real; no calm-window
  counterpart exists to control against.
- **Whether anything changes in a shock window that *fell*.** We have exactly one
  shock window and it rose. Nothing here separates "the agent helps in shocks" from
  "a positively-leaning signal is flattered by a rising market". This is now the
  largest gap in the evidence base.

**The recurring trap** — four times, an apparent agent contribution vanished under a
control:

| looked like | control applied | result |
|---|---|---|
| width channel beats no-adjustment, all CIs exclude 0 | vs. its own constant | identical |
| `w_loc` sweep improves monotonically to 1.0 | other regime | optimum at 0.0 |
| advantage rises monotonically with volatility | within one window | reverses to negative |
| news baseline has the best CRPS | decompose the ingredients | it is the interval width |

Every one showed a clean monotone relationship and passed a significance test. The
rule this yields: an accepted change must clear **four** bars — beats no adjustment,
beats its own **constant**, beats the **unconditional** version, and holds in a
**different regime**.

**Post-rebuild note (2026-08-09).** Re-running this notebook against the rebuilt
news cache (`new_context/`, aligned cutoff) and the anchor-gap fix (§1) reproduced
every structural finding above unchanged in direction — the sign flip, the opposite
`w_loc` optima, the width-channel-is-a-constant result, and the rectification-is-
2026-specific finding (§6) all replicate. The one claim that moved is §2's ③ (news
reasoning "significantly harmful" on preview), which is now a non-significant
positive point estimate instead. See
`planning-docs/news-cache-rebuild-interview-notes.md` for the full before/after
comparison across all 16 arms, and `anchor_2026_eval_scorecard.ipynb` §6/§9 for a
substantially larger movement in the single-window scorecard: the 2026-03-02 "one
day explains the whole gap" finding reversed from "tied once excluded" to "anchored
agent wins once excluded," because the old cached briefing for that origin leaked
the weekend shock it was supposed to be forecasting (same-day cutoff), and the
rebuilt cutoff removes that leak.</cell>
